In [1]:
## 대사 추출하기


In [2]:
file_dir = './_data/'

In [12]:
with open('./_data/0010/고양이를_부탁해_시나리오.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())
    
        

In [16]:
import re
import csv
import io

def extract_dialogue_from_txt(script_text):
    """
    주어진 스크립트 텍스트(.txt)에서 대사만 추출하여 리스트로 반환합니다.
    형식: 이름이 한 줄에 나오고 다음 줄부터 대사가 나오는 형식 및
          이름 : 대사 형식 등을 처리하려고 시도합니다.

    Args:
        script_text (str): 영화 대본 텍스트.

    Returns:
        list: [{'Character': 이름, 'Dialogue': 대사}, ...] 형식의 딕셔너리 리스트.
              추출 실패 시 빈 리스트 반환.
    """
    dialogues = []
    lines = script_text.strip().split('\n')

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # 비어있거나 씬 넘버(#숫자.) 인 경우 건너뜀
        if not line or line.startswith('#') or re.match(r'^#\d+\.', line):
            i += 1
            continue

        # 이름 후보: 주로 한글, 짧은 길이, 콜론(:) 없음, 괄호 설명 가능성 (V.O) 등
        # 이 부분은 대본 형식에 따라 매우 가변적이므로, 완벽하지 않을 수 있습니다.
        # 여기서는 이름이 단독 라인에 있을 가능성을 먼저 봅니다.
        potential_name_match = re.match(r'^([가-힣]+(?:\s*\([^)]+\))?)$', line)

        if potential_name_match and (i + 1 < len(lines)):
            character_name = potential_name_match.group(1).strip()
            # 다음 줄이 대사인지 확인 (빈 줄이 아니고, 다른 이름/씬넘버 패턴이 아닌 경우)
            next_line_index = i + 1
            while next_line_index < len(lines) and not lines[next_line_index].strip():
                next_line_index += 1 # 빈 줄 건너뛰기

            if next_line_index < len(lines):
                next_line = lines[next_line_index].strip()
                is_next_line_name = bool(re.match(r'^([가-힣]+(?:\s*\([^)]+\))?)$', next_line))
                is_scene_heading = next_line.startswith('#') or re.match(r'^#\d+\.', next_line)

                # 다음 줄이 이름이나 씬넘버가 아니라면 대사로 간주
                if not is_next_line_name and not is_scene_heading:
                    current_dialogue_lines = []
                    dialogue_line_index = next_line_index
                    while dialogue_line_index < len(lines):
                        dialogue_part = lines[dialogue_line_index].strip()
                        # 다음 줄이 새로운 이름/씬넘버/빈줄이 아니면 대사 연속으로 간주
                        is_dialogue_name_follow = bool(re.match(r'^([가-힣]+(?:\s*\([^)]+\))?)$', dialogue_part))
                        is_scene_heading_follow = dialogue_part.startswith('#') or re.match(r'^#\d+\.', dialogue_part)

                        if not dialogue_part or is_dialogue_name_follow or is_scene_heading_follow:
                            break # 대사 끝

                        current_dialogue_lines.append(dialogue_part)
                        dialogue_line_index += 1

                    if current_dialogue_lines:
                        full_dialogue = " ".join(current_dialogue_lines).strip()
                        # 괄호 안 지문/설명 제거 (예: (웃으며) 제거)
                        cleaned_dialogue = re.sub(r'\([^)]*\)', '', full_dialogue).strip()
                        # 여러 공백을 하나로 변경
                        cleaned_dialogue = ' '.join(cleaned_dialogue.split())

                        if cleaned_dialogue:
                            dialogues.append({
                                'Character': character_name,
                                'Dialogue': cleaned_dialogue
                            })
                        i = dialogue_line_index # 다음 처리할 라인 인덱스 업데이트
                        continue # 다음 루프로

        # 이름 : 대사 형식도 확인
        dialogue_colon_match = re.match(r'^\s*([^:\n]+?)\s*:\s*(.*)', line)
        if dialogue_colon_match:
             character_name = dialogue_colon_match.group(1).strip()
             dialogue_text = dialogue_colon_match.group(2).strip()
             # 괄호 안 지문/설명 제거
             cleaned_dialogue = re.sub(r'\([^)]*\)', '', dialogue_text).strip()
             cleaned_dialogue = ' '.join(cleaned_dialogue.split())
             if cleaned_dialogue:
                 dialogues.append({
                     'Character': character_name,
                     'Dialogue': cleaned_dialogue
                 })
             i += 1
             continue # 다음 루프로

        # 위 패턴에 맞지 않으면 다음 줄로 이동
        i += 1

    return dialogues


In [ ]:

# 3. 대사 추출 함수 호출
extracted_dialogues = extract_dialogue_from_txt(script_content)

In [ ]:


# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("고양이를_부탁해_대사.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '고양이를_부탁해_대사.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")

In [17]:
with open('./_data/0010/공동경비구역JSA_시나리오.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_from_txt(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("공동경비구역JSA_시나리오.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '공동경비구역JSA_시나리오.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"밤. 비가 내린다. 둥지에 숨어 눈만 끔뻑이는 올빼미. 먹구름 낀 하늘. 화면 상단에 적절한 기계 음과 함께 자막. 02","24, Nov 18. 판문점 공동경비구역 주위의 숲 풍경. 다리 난간에 떨어지기 시작하는 빗방울. 들릴 듯 말 듯 흐르는 음악, 김광석이 부르는 [이등병의 편지]. 다리와 두 초소의 전경 위로 점점 거세게 쏟아지는 빗줄기. 검게 가려진 북측 초소의 유리창 클로즈업. 돌연 총성과 함께 창에 구멍이 뚫린다. 총구멍을 통해 안으로부터 노란 전등 빛이 새어나오면서 빗줄기를 비춘다. 총성에 놀라 홱 고개를 돌리는 올빼미, 불안하게 움직이는 노란 눈동자 두 개의 클로즈업."
"화면 하단에 14","05, Nov 18."
"중앙방송","방금 발표된 중앙군사위원회의 성명을 읽어드리겠습니다."
"대변인",".... 오늘 새벽 공두 시 이십 오 분...."
"중앙방송","....초소에 남조선 괴뢰군의 가증스러운 기습 테러 공격이.... ....있었다. 이 사고로...."
"대변인","북괴군 두 명이 죽고 한 명이 부상을 당했습니다...."
"대변인","....초소 경계 근무 중이던 한미연합사 소속 이수혁 병장은 군사분계선을 넘어 침투한 북괴군에 의해 납치되어 감금되었다가.... ....탈출하는 과정에서 이 같은 총격전을 벌인 것으로 보입니다."
"중앙방송","....그러나 경애하는 최고사령관 동지의 지도를 꿈에도 잊지 않은 우리의 오경필 중사는.... ....단신으로 용맹히 응전하여 적을 부상 입혀...."
"대변인","....군사분계선 위에 쓰러진 채 발견됐습니다...."
"중앙방송","....우리 인민은 주체조선, 영웅조선의 명예를 걸고, 계속되는 미제와.... .... 남조선 괴뢰도당의 노골적인 전쟁 기도에 대해 백 배 천 배의 복수를 다짐하면서...."
"대변인","....더 이상의 어떠한 도발도 좌시하지 않을 것임을 엄중히 경고하는 바입니다."
"대변인","....이상입니다."

In [19]:
def extract_dialogue_tab_format(script_text):
    """
    주어진 스크립트 텍스트에서 대사만 추출하여 리스트로 반환합니다.
    형식: '이름\t대사' 및 '\t대사' (연속 줄) 형식을 처리합니다.

    Args:
        script_text (str): 영화 대본 텍스트.

    Returns:
        list: [{'Character': 이름, 'Dialogue': 대사}, ...] 형식의 딕셔너리 리스트.
              추출 실패 시 빈 리스트 반환.
    """
    dialogues = []
    lines = script_text.strip().split('\n')

    current_character = None
    current_dialogue_lines = []

    # 이름 TAB 대사 패턴
    dialogue_start_re = re.compile(r'^([^\t\n]+)\t(.*)')
    # TAB 대사 (연속) 패턴
    continuation_re = re.compile(r'^\t(.*)')
    # 씬 넘버 패턴 (숫자. 으로 시작)
    scene_re = re.compile(r'^\d+\.\s+')
    # 괄호 안의 지문/설명 제거 패턴
    action_re = re.compile(r'\([^)]*\)')

    for line in lines:
        line_stripped = line.strip() # 앞뒤 공백 제거 후 내용 확인용
        
        # 빈 줄 또는 씬 넘버 건너뛰기
        if not line_stripped or scene_re.match(line):
            # 이전 대사가 있었다면 저장
            if current_character and current_dialogue_lines:
                full_dialogue = " ".join(current_dialogue_lines).strip()
                cleaned_dialogue = action_re.sub('', full_dialogue).strip()
                cleaned_dialogue = ' '.join(cleaned_dialogue.split())
                if cleaned_dialogue:
                    dialogues.append({
                        'Character': current_character,
                        'Dialogue': cleaned_dialogue
                    })
            current_character = None
            current_dialogue_lines = []
            continue

        match_start = dialogue_start_re.match(line) # 원본 라인으로 매칭 (탭 확인 위해)
        match_cont = continuation_re.match(line) # 원본 라인으로 매칭 (탭 확인 위해)

        if match_start:
            # 이전 대사가 있었다면 저장
            if current_character and current_dialogue_lines:
                full_dialogue = " ".join(current_dialogue_lines).strip()
                cleaned_dialogue = action_re.sub('', full_dialogue).strip()
                cleaned_dialogue = ' '.join(cleaned_dialogue.split())
                if cleaned_dialogue:
                    dialogues.append({
                        'Character': current_character,
                        'Dialogue': cleaned_dialogue
                    })
            
            # 새 대사 시작
            current_character = match_start.group(1).strip()
            initial_dialogue = match_start.group(2).strip()
            current_dialogue_lines = [initial_dialogue] if initial_dialogue else []

        elif match_cont and current_character:
            # 대사 연속
            continuation_text = match_cont.group(1).strip()
            if continuation_text:
                current_dialogue_lines.append(continuation_text)
                
        else: # 지문 또는 다른 형식의 라인
            # 이전 대사가 있었다면 저장
            if current_character and current_dialogue_lines:
                full_dialogue = " ".join(current_dialogue_lines).strip()
                cleaned_dialogue = action_re.sub('', full_dialogue).strip()
                cleaned_dialogue = ' '.join(cleaned_dialogue.split())
                if cleaned_dialogue:
                    dialogues.append({
                        'Character': current_character,
                        'Dialogue': cleaned_dialogue
                    })
            current_character = None
            current_dialogue_lines = []

    # 마지막 대사 블록 처리
    if current_character and current_dialogue_lines:
        full_dialogue = " ".join(current_dialogue_lines).strip()
        cleaned_dialogue = action_re.sub('', full_dialogue).strip()
        cleaned_dialogue = ' '.join(cleaned_dialogue.split())
        if cleaned_dialogue:
            dialogues.append({
                'Character': current_character,
                'Dialogue': cleaned_dialogue
            })

    return dialogues

In [23]:
with open('./_data/9000/건축학개론_12.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_tab_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("건축학개론_12.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '건축학개론_12.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"승민","아... 결려."
"구소장","아 자식. 밤 샌 거 꼭... 이렇게 극적으로 티를 내요. 하튼 보면 영리해. 찜질방이라도 갔다 와. 미팅 2시니까..."
"승민","내가 건축주 만나 뭐해요?"
"구소장","임마. 디자인 한 놈이 같이 가야지."
"승민","... 나 옷도 없는데..."
"구소장","양복 하나 갖다 놓으랬지!"
"은채","아 진짜! 사무실에서 피지 말라니깐! 간접흡연이 얼마나 안좋은데!"
"승민","그럼 직접흡연하는 난 얼마나 안좋겠니? 안그래? 보면 꼭... 지 생각만 하고."
"은채","손님 왔어요."
"승민","손님? 누구?"
"은채","몰라요. 팀장님 친구래요."
"승민","내 친구? 친구 누구?"
"은채","그걸 내가 어떻게 알아요!"
"서연","오랜만이네."
"승민","..."
"서연","동문주소록 정확하네. 혹시나 했는데. 근데 맞네."
"승민","저기... 근데..."
"서연","... ?"
"승민","... 누구... 신지?"
"서연","나 몰라? ...... 세요?"
"승민","... ?"
"서연","저기... 옛날에... 대학 1학년때..."
"서연","어떻게 날 까먹어?"
"승민","에이. 까먹긴 뭘 까먹어. 하도 오랜만이니까..."
"서연","그래두. 나 별루 변한 것도 없는데."
"승민","살은 좀... 찐 거 같은..."
"서연","..."
"승민","그래서... 무슨 일 해?"
"서연","그냥 방송일 조금."
"승민","방송? 방송 뭐?"
"서연","케이블 같은데서 아침에 배도 타고 산도 오르고. 있어 그런 거."
"승민","사는 덴 어디야?"
"서연","개포동."
"승민","남편은 뭐하는데?"
"서연","... 동사무소에서 인구조사 나왔니? 지금."
"서연","옛날에 살던 집인데... 너무 낡아서. 이번에 싹 밀고 제대로 새로 지으려고."
"승민","..."
"서연","언제부터 시작할 수 있어?"
"승민","나? ... 나보고 하라고?"


In [21]:
with open('./_data/0010/미녀는 괴로워.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_from_txt(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("미녀는 괴로워.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '미녀는 괴로워.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"선녀보살","자주 온다고 운명이 바뀌나, 응? 그래서 관상이 바뀌고 팔자가 돌아간다면야 나두 신이 나서 하겠어. 아니잖아? 다시 한 번 얘기해줘? 힘들게 살아왔고, 앞으로도 쭉 힘들거야! 그리고, 이 남자하곤 절대 안돼. 될 수가 없어. 나두 좋은 얘기해주고 싶어. 근데 어쩌겠어. 관상이 그런데?"
"누군가","그래도...사람 일은 모르는 거잖아요."
"선녀보살","그러니까! 그걸 왜 굳이 알려구 그래. 참나... 내가 부적 하나 써 줄 테니까 그 사람 몸에 지니게 해. 잘 된단 보장은 없지만... 돈은 됐구, 이거 받구 얼른 가봐."
"선녀보살","왜, 왜? 야! 야! 하지마!!"
"한나","절 올릴려구요..감사해서요..제 성의에여.."
"선녀보살","더해? 더~ 응? 이거 안 파는건데.. 참 궁금해..너 도데체 뭐 잘하니? 민망해 하던 한나, 고개 들어 묘하게 보살을 쳐다보며 미소 짓는다."
"한나","어! 그만, 그만... 됐어요. 오늘은 그만해요. 국사선생님 들어오세요. 아!! 그만... 오늘 오랄박 너무 짓궂어요."
"한나(N)","아름다운 목소리로 왜 이런일을 하냐구요? 이 직장은 면접이 없으니까요. 그래서 떨어질 일도 없죠. 너무 이상하게 생각하지 마세요. 저한테 전화거는 사람들요..모두 다 외로운 사람들이랍니다. 전 그런 사람들 위로해 주는 거구요."
"통화남(E)","너무 외로워..나란놈..의사도 아니고, 좋은 남편도 못되고.."
"통화남(E)","나한텐 너 밖에 없다.. 자, 시작하자. 오늘은 김 간호사로 가자."
"한나","저 지금 흠뻑 젖어있어요~ 제발 나 좀.. 말려주세요~ 네?"
"한나","난 가수랍니다!!"
"타이틀","미녀는 괴로워"
"CUT TO","무대 뒤 아무도 모르는 곳, 작은 부스 안을 꽉 메우고 선 한나."
"한나 (N)","얼굴 없는 가수가 어때서요? 진짜 가수는 난데."
"한나","한상준. 나에게 삶의 이유를 느끼게 해준 유일한 사람."
"한나 (N

In [22]:
with open('./_data/0010/엽기적인_그녀.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_tab_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("엽기적인_그녀.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '엽기적인_그녀.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"﻿#1.","강가, 기차길"
"#2.","강과 기차길이 내려다보이는 나무 아래"
"견우","2년 전 바로 오늘, 그녀와 저는 이 자리에 타임캡슐을 무더씀미다. 오늘은 우리가 2년만에 다시 만나는 날이지만 그녀는 아직 나타나지 안씀미다. 전 기다립미다."
"#3","사진관"
"사진사","하나, 둘..."
"견우","잠깐만여... 여보세요...? 네? 고모? 예... 죄송해요... 갈께요. 죄송하다고 했잖아요... 네... 간다니까여? 사진 찍고 있어요... 네."
"견우(나레이션)","...부모님은 제가 딸이길 원해서 저는 어려서 부텀 딸처럼 키우셔씀미다."
"견우(나레이션)","그래서 저는 일곱 살까지 여잔줄로만 알았씀미다. 글구 목욕탕도 엄마하고만 갔씀미다. 저는 나이가 들면 꼬추가 점점 작아져서 사라지는 줄로만 알았씀미다. 근데, 정 반대더군여."
"#4.","순대집"
"견우(나레이션)","...저는 군대생활을 무사히 끝내고 복학을 했씀미다."
"친구들-","-야, 띱때야! 공익근무요원이 무슨 군대생활이냐? 제대 조아하네?"
"견우","띱때야, 공익근무요원이 뭐냐? 공근이라고 불러! 공근! 이래뵈도 전방에서 근무했단말야!"
"친구들","-공근은 구파발이 전방이냐? -띱때야, 너는 제대한 게 언젠데 인제 연락하고 질알이냐? -어째뜬 견우가 무사히 제대한 것과 복학을 추카한다! ...건배!"
"견우","제 이상형임미다. 이상형이 지나가면 저는 못참씀미다. 말을 부쳐바야져!"
"견우","에이씨! 중요한 시간에 ...여보세요? 누구냐?"
"견우의 母","니 엄마닷! 너 고모네 간다더니 지금 뭐하고 있는 거얏!"
"견우","곧 갈 껀대여? 조용히 해 띱때들아. 엉아 저나 받잖아! .\ /."
"견우의 母","오늘은 꼭 좀 갔다 와라, 응? 고모 본 지 너 1년도 넘어찌?"
"견우","작년에 반나?"
"견우의 母","고모 작년에 하나밖에 없는 자식 잃고 적적하게 사는 거 잘 알잖아... 너하구 걔 

In [25]:
def extract_dialogue_colon_format(script_text):
    """
    주어진 스크립트 텍스트에서 대사만 추출하여 리스트로 반환합니다.
    형식: '이름 : 대사' 형식을 주로 처리합니다.

    Args:
        script_text (str): 영화 대본 텍스트.

    Returns:
        list: [{'Character': 이름, 'Dialogue': 대사}, ...] 형식의 딕셔너리 리스트.
              추출 실패 시 빈 리스트 반환.
    """
    dialogues = []
    lines = script_text.strip().split('\n')

    # 이름 : 대사 패턴 (이름에 괄호 설명 포함 가능)
    dialogue_re = re.compile(r'^\s*([^:\n]+?)\s*:\s*(.*)')
    # 씬 넘버 패턴
    scene_re = re.compile(r'^S#\s*\d+')
    # 괄호 안의 지문/설명 제거 패턴
    action_re = re.compile(r'\([^)]*\)')

    for line in lines:
        line_stripped = line.strip()

        # 빈 줄 또는 씬 넘버 건너뛰기
        if not line_stripped or scene_re.match(line):
            continue

        match = dialogue_re.match(line)
        if match:
            character_name = match.group(1).strip()
            dialogue_text = match.group(2).strip()

            # 이름에 포함된 (V.O), (off) 등은 유지할 수 있으나,
            # 대사 중의 (놀란듯), (엔지니어에게) 등은 제거
            cleaned_dialogue = action_re.sub('', dialogue_text).strip()
            # 여러 공백을 하나로 정리
            cleaned_dialogue = ' '.join(cleaned_dialogue.split())

            if cleaned_dialogue: # 내용이 있는 경우만 추가
                dialogues.append({
                    'Character': character_name,
                    'Dialogue': cleaned_dialogue
                })
        # else:
            # 연속 줄 처리 로직 추가 가능 (들여쓰기 등 기준 필요)
            # 이 예제에서는 '이름 : 대사' 형식만 처리합니다.

    return dialogues

In [26]:
with open('./_data/9000/8월의_크리스마스.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_colon_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("8월의_크리스마스.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '8월의_크리스마스.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"소리","열중 쉬어. 차렷, 교장선생님께 대하여 경례. 바로 열중 쉬어"
"다른소리","이어서 애국가 재창이 있겠습니다. 애국가는 1절부터 4절까지."
"소리","전체 차렷."
"여자","아저씨, 얼굴이 좀 이상하게 나왔어요."
"다림","번호판이 하나도 한 보이네."
"다림","왜 웃어요?"
"다림","할 수 없죠 뭐. 아저씨 필름 좀 넣어 주세요."
"정원","필름을 넣을 줄 몰라요 ?"
"다림","좀 해주세요."
"정원","이리 와봐요. 가르쳐 줄테니 잘 들어요"
"정원","설명은 듣지 않고.. 아가씨, 셔터 누를 때 숨을 멈춰."
"철구","니가 알아서 잘 좀 골라 봐.. 이따가 병원에서 연락할께."
"철구","아버지 친구분들이 영정 사진보고 막 웃으시더라구 예날 생각이 났나봐"
"정원","너무 젊었을 때 사진을 썼나 ?"
"철구","아니야 한참 잘나갈 때 사진이라서 당신도 좋아하셨을거야"
"정원","여기서 뭐해요?"
"다림","한참 기다렸어요."
"정원","...."
"정원","이따가 오면 안돼요 ?"
"다림","이거 급한거니까 빨리 찾아오래요. 얼마나 걸려요 ?"
"정원","아까 저 때문에 화났었죠 ?"
"다림","아침부터 혼나고, 너무 더워서 그래요."
"정원","이리와 앉아요."
"다림","더운건 정말 싫어요"
"정원","난 좋은데"
"다림","이번 사진은 어떻게 나올지..걱정된다. 이번에도 촛점이 안맞으면"
"아버지","뭐하다가 이렇게 늦게 들어오냐."
"중학생 1","여기있는 애가 내가 찍은 애예요."
"중학생 2","이 애는 내꺼예요."
"중학생 2","뻥가고 있네. 이게 어디 예쁘냐? 아저씨 얘가 더 예쁘죠?"
"정원","이 걸 확대하면 좀 흐리게 나올텐데."
"중학생 1","괜찮아요."
"정원","너는?"
"중학생 3","난 좀 많아요."
"중학생 1","너 내꺼 건드리지마."
"정원","야 야. 싸우지들 마라. 너희들 이 여학생들한테 말이나 걸어 봤어?"
"정원","

In [27]:
with open('./_data/9000/초록 물고기.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_colon_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("초록 물고기.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '초록 물고기.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"불량배1","이럴거 없잖아, 같이 놀자는데"
"막동이","왜 그래요?"
"불량배1","뭘?"
"막동이","왜 가만 있는 사람을 괴롭혀요?"
"불량배2","야, 이 …… 시발 니가 뭔데 나서?"
"불량배3","괴롭히기는 뭘 괴롭혀?"
"막동이","그럼 됐어요.."
"불량배1","되긴 뭐가 돼?"
"막동이","엄마!"
"막동이","엄마,"
"막동이","둘째 형 하는 일은 잘돼?"
"어머니","참, 니 큰성이 아침부터"
"막동이","예, 통신보안 김병"
"어머니","이리 줘 봐라."
"어머니","여보세요, 예 맞아요."
"막동이","엄마. 파출부"
"막동이","엄마! 이제 내가 돈 많이 벌테니까"
"어머니","참, 어제 너 찾는 전화가 왔었는데."
"막동이","누군데?"
"어머니","내가 아니? 응, 거기 어디"
"막동이","(전화기 옆에 씌여진 먼호를 보며 전화"
"막동이","안받네."
"셋째","그래, 제대 소감이 어떠셔?"
"막동이","답답하지 뭐. 할 일도 없고…"
"셋째","나야 뭐 먹구 살기 바쁘지 뭐."
"막동이","동네가 너무 많이 변했어. 신도시"
"셋째","전에는 뭐 별 거 있었나?"
"막동이","그래도......"
"셋째","아, 계란이요, 계란! 계란이"
"막동이","미애씨요?"
"막동이","형님, 여기가 옛날 우리땅 아냐?"
"막동이","옛날 여기 아카시아 천지였는데......"
"셋째","임마, 이제 너도 돈벌 궁리나 해. 니"
"막동이","아이, 알았어. 돈 벌거야. 두고봐, 어"
"셋째","아, 씨발 걸렸네!"
"막동이","왜 안 서, 도망가면 더 크게 걸려."
"셋째","가만 있어."
"교통","실례합니다."
"셋째","아,"
"교통","아무리 바빠도 신호는 지키셔야지."
"셋째","한번만 봐주세요. 서로 아는 처지에…"
"교통","알긴 뭘 알아?"
"셋째","딱지 떼면 오늘 장사 공쳐요. 나 하루"
"막동이","아저씨 한 번만 봐줘요."
"셋째","아이 봐줘요."


In [28]:
with open('./_data/9000/접속.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_colon_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("접속.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '접속.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"각본","조명주 / 장윤현"
"진행자","비도 오는데 뭐 찡! 한 음악 없을까요?"
"은  희","오늘은 신청곡 좀 많이 받을까요?"
"동  현","비하고 관계있는 것들로 골라봐"
"진행자","오늘 20분짜리 곡나가요?"
"동  현","2부에 20분짜리 곡 나가니까 전 CM을 조금 빨리"
"엔지니어","20분... 너무긴데..."
"태  호","Good News, Bad News 뭐부터 들을래?"
"동  현","Good News도 있어요?"
"태  호","Good News는 어제 음악선곡 완벽했다는 거고..."
"동  현","5분짜리 노래만 선곡할거면 PD가 왜 필요합니까?"
"태  호","5분짜리만 선곡하라고 있는거야!"
"동  현","노래 한 곡이 그렇게 심각한 겁니까?"
"태  호","원래 윗사람들이란게 개편때만 심각하잖니."
"동  현","심야프로 질 높이라면서도 청취율은 포기 못하죠."
"태  호","타협하기 싫다고 프로 없앨거야?"
"동  현","장수 프로 만들려고...색깔을 죽일 수는 없어요."
"태  호","그래. 나도 너말할땐 그랬어. (알았다는 듯 고개를 끄덕"
"은  희","그것도 안 된대요? 촌스러워 정말..."
"은  희","...어떻게 됐어요?"
"동  현","매일 부딪치는 일이야. 신경쓰지마."
"은  희","다음 개편때 없어진다는 말이 있어요."
"동  현","개편전엔 늘 그래. 왜 걱정돼?"
"은  희","제가 맡은지 한달밖에 안됐어요. 작가한텐 좋지못한 경력이에요."
"동  현","자료실에 있을거야."
"은  희","PD가 잘리는 경우란 없으니까..."
"은  희","잠깐만요!"
"은  희","어떤 여자 분이 수위실에 맡겨놨대요. 드리면 아실거라고 했다는"
"동  현","이게 다야? 다른 거 뭐 없었어?"
"은  희","뭐가 더 필요해요?"
"수  현","엄마야! 기철씨...! 왜 그래?"
"기  철","놀래라고 한거다...아무튼 말 되게 안들어? 이렇게 구석

In [30]:
with open('./_data/8090/살인의_추억.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_tab_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("./_data/8090/살인의_추억.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '살인의_추억.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"형사","박두만 서태윤 조용구"
"여경","권귀옥"
"반장1","구희봉"
"반장2","신동철"
"용의자","백광호 20대 초반. 정박아 조병순 30대 후반. 변태성향 박해일 20대 초반. 공장 노동자. ‘유력한 용의자’."
"그 외 용의자들","선본 남자, 동네 양아치들... 등등"
"곽설영","30대 초반의 전직 간호조무사, 마을 ‘야매주사’ 여인."
"김소현","안송여중 1학년 학생"
"오남주","소현의 단짝 친구"
"피해자들","박보희, 이향숙, 독고현순, 박명자, 안미선."
"두만","너 밤길에 이 여자한테 뎀빈게 작년 8월 아냐, 맞어 틀려?"
"용의자1","맞는데요 ..."
"두만","이 쉐끼가 .."
"두만","그날 박보희씨와 선을 보고 헤어진 게 언제쯤이셨죠?"
"양복","같이 저녁 먹고 ... 7시 반쯤요 ?"
"두만","그 이후엔 어디 계셨습니까 ?"
"두만","눈감았지 ?"
"두만","움직이지 말어 !"
"두만","와 ... 진짜 인상들 좇같네 ..."
"구반장","... 그딴걸 뭣하러 다 모아놨는겨 ?"
"두만","조카들 얼굴을 자꾸 보다보면 ... 어느 순간 딱 느낌이 오거든요."
"두만","딴건 몰라도 내가 사람 얼굴 보는 재주는 있다는 거 아닙니까."
"구반장","아예 돗자리를 깔지 그려 ?"
"두만","정말요 ! 사람 얼굴을 딱 - 들여다 보면 이새끼가 착한놈인지 나쁜놈인지, 죄가 있는지 없는지, 금방 답이 나와요. 내 눈은 못속이거던 ... 그래서 제가 이 순사밥 먹는거 아니겠슴까 ?"
"구반장","구라하고는 ... 그럼 저기 ... 쟤네들 한번 봐 봐. 구석에 나란히 앉은 두 놈 보이는겨 ?"
"두만","네 ..."
"구반장","저중에 한 놈은 강간범, 완전 꽈배기고 ... 한명은 피해자 오빠여. 그러니까 피해자 오빠가 지 여동생 강간한 새끼를 지 손으루 잡아가지고 왔다 이거지."
"구반장","어느쪽이 강간인지 ... 알, 아, 맞, 춰, 보, 시, 요 ...

In [72]:
import re

def extract_dialogue_indented_final_v3(script_text):
    """
    주어진 스크립트 텍스트에서 대사만 추출하여 리스트로 반환합니다. (들여쓰기 활용 및 연속줄 처리 개선)
    형식: '    이름    대사' 형식 및 연속 줄 대사를 처리합니다.
          continuation_indent_re 변수를 실제로 활용합니다.

    Args:
        script_text (str): 영화 대본 텍스트.

    Returns:
        list: [{'Character': 이름, 'Dialogue': 대사}, ...] 형식의 딕셔너리 리스트.
    """
    dialogues = []
    # 원본 줄바꿈 유지, 페이지 넘김 문자는 제거
    lines = script_text.replace('\f', '').split('\n')

    # 이름+대사 시작 패턴: ^(들여쓰기)(이름)(공백+)(대사)
    dialogue_start_re = re.compile(r'^(\s+)([^\s].*?[^\s](?:\s*\([^)]+\))?)\s+(.+)')
    # 이름 없이 들여쓰기만 된 연속 대사/지문 패턴
    continuation_indent_re = re.compile(r'^(\s+)(.*)') # 들여쓰기 된 모든 줄
    # 씬 넘버 패턴
    scene_re = re.compile(r'^\d+\.\s+')
    # 괄호 안 지문 제거 패턴
    action_re = re.compile(r'\([^)]*\)')
    # 빈 줄 확인용
    empty_line_re = re.compile(r'^\s*$')

    current_character = None
    current_dialogue_lines = []
    last_indent_level = -1 # 마지막 대사 시작 줄의 들여쓰기 레벨

    for line in lines:
        stripped_line = line.strip()

        # 빈 줄 처리
        if not stripped_line:
            if current_character and current_dialogue_lines:
                full_dialogue = " ".join(current_dialogue_lines).strip()
                full_dialogue = ' '.join(full_dialogue.split())
                if full_dialogue:
                    dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})
            current_character = None
            current_dialogue_lines = []
            last_indent_level = -1
            continue

        # 씬 넘버 처리
        if scene_re.match(stripped_line):
            if current_character and current_dialogue_lines:
                 full_dialogue = " ".join(current_dialogue_lines).strip()
                 full_dialogue = ' '.join(full_dialogue.split())
                 if full_dialogue:
                     dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})
            current_character = None
            current_dialogue_lines = []
            last_indent_level = -1
            continue

        match_start = dialogue_start_re.match(line) # 이름 + 대사 시작 확인

        if match_start:
            # 이전 대사 저장
            if current_character and current_dialogue_lines:
                full_dialogue = " ".join(current_dialogue_lines).strip()
                full_dialogue = ' '.join(full_dialogue.split())
                if full_dialogue:
                    dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})

            # 새 대사 시작
            current_indent = len(match_start.group(1))
            current_character = match_start.group(2).strip()
            initial_dialogue = match_start.group(3).strip()
            cleaned_part = action_re.sub('', initial_dialogue).strip()
            current_dialogue_lines = [cleaned_part] if cleaned_part else []
            last_indent_level = current_indent

        elif current_character: # 현재 화자가 있는 상태에서
            cont_match = continuation_indent_re.match(line) # <<-- continuation_indent_re 사용
            if cont_match:
                 current_line_indent = len(cont_match.group(1))
                 # 이전 대사 시작 줄보다 들여쓰기가 같거나 크고, 내용이 있다면 연속 대사로 간주
                 if current_line_indent >= last_indent_level:
                     continuation_text = cont_match.group(2).strip()
                     # 혹시 들여쓰기된 줄이 이름 패턴으로 시작하면 제외 (지문 등)
                     if not re.match(r'^[^\s].*?[^\s](?:\s*\([^)]+\))?\s+', continuation_text):
                         cleaned_part = action_re.sub('', continuation_text).strip()
                         if cleaned_part:
                             current_dialogue_lines.append(cleaned_part)
                     else: # 들여쓰기 됐지만 새 이름 패턴이면 이전 대사 저장
                          if current_character and current_dialogue_lines:
                              full_dialogue = " ".join(current_dialogue_lines).strip()
                              full_dialogue = ' '.join(full_dialogue.split())
                              if full_dialogue:
                                  dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})
                          current_character = None
                          current_dialogue_lines = []
                          last_indent_level = -1
                 else: # 들여쓰기가 이전보다 작으면 지문으로 보고 이전 대사 저장
                     if current_character and current_dialogue_lines:
                          full_dialogue = " ".join(current_dialogue_lines).strip()
                          full_dialogue = ' '.join(full_dialogue.split())
                          if full_dialogue:
                              dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})
                     current_character = None
                     current_dialogue_lines = []
                     last_indent_level = -1
            else: # 들여쓰기 없는 줄은 지문으로 보고 이전 대사 저장
                if current_character and current_dialogue_lines:
                     full_dialogue = " ".join(current_dialogue_lines).strip()
                     full_dialogue = ' '.join(full_dialogue.split())
                     if full_dialogue:
                         dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})
                current_character = None
                current_dialogue_lines = []
                last_indent_level = -1
        # else: # 이름 패턴도 아니고, 현재 화자도 없으면 지문 (처리 없음)
        #     pass

    # 마지막 대사 블록 처리
    if current_character and current_dialogue_lines:
        full_dialogue = " ".join(current_dialogue_lines).strip()
        full_dialogue = ' '.join(full_dialogue.split())
        if full_dialogue:
            dialogues.append({'Character': current_character, 'Dialogue': full_dialogue})

    return dialogues

In [73]:
with open('./_data/8090/써니.txt', 'r', encoding='utf-8') as f:
    script_content = f.read()

# print(script_content)
extracted_dialogues = extract_dialogue_indented_final_v3(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("./_data/8090/써니.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '써니.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

--- CSV 출력 ---
"Character","Dialogue"
"각본/감독","강형철"
"제  작","토일렛 픽처스"
"만드는","음식들은 한식, 건강식, 서양식등 다양하다. 대강 분위기로 보아 좀 사는 집인 듯."
"주름이","생겼나 자세히 들여다보는 모습이 귀엽기까지 하다."
"주방","라디오에서는 여전히 time after time 이어지고 신문 보는 남편은 복 해장국,"
"문자","날리는 딸은 서양식 아침. 대화는 없다."
"옆에서","열심히 이들의 식사를 시중드는 주부 나미."
"(그냥","가버리는 딸. 다시 남편에게) 응?"
"신발을","신고 지갑에서 상품권 봉투를 꺼내는 남편."
"신경","접고 고만 일 보소. 번창하시게. 잉. 잉. 그려 그려..들어가소. 어이."
"(끊고","가방을 들어 보이며 환자들에게) 우리 사위가."
"엄마는","뿔이 났어도 사랑이 뭐길래? 아들과 딸을 둔 죄로"
"완전한","사랑을 요구 받는 게 애민게지."
"알았어요.","지금 들어갈게.."
"(말자)","저기 이번에 전학 온 임..나미. 임 나미 학생이다."
"전라도","벌교에서 왔고 서울은 처음이니까 다들 잘 해주고."
"(답답)","응? 장미야! 자기소개 해."
"꼬막","나는 동네. 도시락 봐바. 꼬막 싸왔냐?"
"천 이백","덕성인들의 활명수 D.J 홍 인사드리면서 날려드리는 오늘의 첫 곡."
"rick","astley오빠의 never~. never gonna give up!"
"(누군가","발견) 어이! 서금옥이! 사라다 빵 네 개 추가 !!"
"다구발만","안 세우면 참 참한데.... 말이지."
"(나미","보며) 얘가 전학생? 반갑다 얘. 나 미스코리아 나가야 되거든."
"아까","우리 반에 예쁜 애 봤지? 피비 케이츠 같이 생긴 애. 수지."
"걔 까지","해서 우리 멤바야. 멤바."
"그거","했는데. 친구들 많이 사겼는가?"
"프로","스팩스만 신더만? 바지도 뱅뱅 산다니깐 죠다쉬 좋다고..."
"잔업을","얼마나 하는지 알기나 해?"
"인권을"

In [64]:
import re
import io
import csv # CSV 출력을 위해 필요

# 여기에 이전에 제공된 extract_dialogue_indented_format_final_v2 함수 정의가 와야 합니다.
# def extract_dialogue_indented_format_final_v2(script_text):
#     ... (함수 내용) ...

# --- 파일 읽어오는 부분 ---

# 1. 파일 경로 지정
#    사용자님이 업로드하신 파일명을 사용합니다.
#    실제 환경에서는 정확한 경로를 지정해야 합니다.
#    (예: './_data/8090/써니.txt' 또는 사용자가 지정한 경로)
file_path = '써니.txt' # 업로드된 파일명을 사용

script_content = None
try:
    # 2. 파일 열기 및 읽기
    #    - 'r' 모드는 읽기 모드입니다.
    #    - 'encoding='utf-8''은 한글 텍스트 파일을 올바르게 읽기 위해 중요합니다.
    #      만약 파일이 다른 인코딩(예: cp949/euc-kr)으로 저장되었다면 해당 인코딩으로 변경해야 합니다.
    with open(file_path, 'r', encoding='utf-8') as f:
        # 3. 파일 내용 전체를 하나의 문자열로 읽기
        script_content = f.read()
    print(f"'{file_path}' 파일을 성공적으로 읽었습니다.")

except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")

# --- 파일 읽기 완료 후 대사 추출 및 처리 ---

if script_content: # 파일 내용을 성공적으로 읽었을 경우에만 실행
    # 대사 추출 함수 호출
    extracted_dialogues = extract_dialogue_indented_format_final_v2(script_content)

    # 4. 결과 확인 (예: CSV 형태로 출력)
    if extracted_dialogues:
        output = io.StringIO()
        writer = csv.writer(output, quoting=csv.QUOTE_ALL)
        writer.writerow(['Character', 'Dialogue']) # 헤더 작성
        # 결과가 너무 길 수 있으므로 일부만 출력 (예: 처음 20개)
        for entry in extracted_dialogues[:20]:
             writer.writerow([entry['Character'], entry['Dialogue']])

        csv_output = output.getvalue()
        output.close()
        print("\n--- CSV 출력 (일부) ---")
        print(csv_output)

        # # 전체 결과를 파일로 저장하고 싶을 경우:
        # full_output = io.StringIO()
        # full_writer = csv.writer(full_output, quoting=csv.QUOTE_ALL)
        # full_writer.writerow(['Character', 'Dialogue'])
        # for entry in extracted_dialogues:
        #     full_writer.writerow([entry['Character'], entry['Dialogue']])
        # csv_full_output = full_output.getvalue()
        # full_output.close()
        # with open("써니_대사추출_결과.csv", "w", encoding="utf-8", newline="") as f:
        #      f.write(csv_full_output)
        # print("\n전체 CSV 파일 '써니_대사추출_결과.csv' 생성 완료.")

    else:
        print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")
else:
    print("스크립트 내용을 읽지 못하여 대사 추출을 진행할 수 없습니다.")

오류: '써니.txt' 파일을 찾을 수 없습니다. 경로를 확인해주세요.
스크립트 내용을 읽지 못하여 대사 추출을 진행할 수 없습니다.


In [60]:
def extract_dialogue_enter_format(script_text):
    """
    주어진 스크립트 텍스트에서 대사만 추출하여 리스트로 반환합니다.
    형식: 이름이 한 줄에 나오고 다음 줄부터 대사가 나오는 형식을 처리합니다.

    Args:
        script_text (str): 영화 대본 텍스트.

    Returns:
        list: [{'Character': 이름, 'Dialogue': 대사}, ...] 형식의 딕셔너리 리스트.
              추출 실패 시 빈 리스트 반환.
    """
    dialogues = []
    lines = script_text.strip().split('\n')

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # 비어있거나 씬 넘버(#숫자. 또는 숫자.) 인 경우 건너뜀
        if not line or line.startswith('#') or re.match(r'^\d+\.\s+', line):
            i += 1
            continue

        # 이름 후보: 주로 한글, 짧은 길이, 콜론(:) 없음, 괄호 설명 가능성 (V.O), (소리) 등
        # 이름으로 간주할 수 있는 패턴 (좀 더 유연하게)
        # - 한글, 영문, 숫자, 공백, 괄호 포함 가능
        # - 너무 길지 않은 라인 (예: 30자 미만)
        # - 명확한 지문 형태가 아닌 경우 (예: '...를 지나치는데-' 로 끝나지 않는 경우)
        potential_name_match = re.match(r'^([^\t\n]{1,30})$', line) # 탭, 개행 제외 1~30자
        is_likely_action = line.endswith('는데-') or line.endswith('다.') or line.endswith('보인다.') or line.endswith('있다.') # 지문일 가능성이 높은 경우 제외

        if potential_name_match and not is_likely_action and (i + 1 < len(lines)):
            potential_character_name = line # 일단 현재 줄을 이름 후보로

            # 다음 줄이 대사인지 확인 (빈 줄이 아니고, 다른 이름/씬넘버 패턴이 아닌 경우)
            next_line_index = i + 1
            while next_line_index < len(lines) and not lines[next_line_index].strip():
                next_line_index += 1 # 빈 줄 건너뛰기

            if next_line_index < len(lines):
                next_line = lines[next_line_index].strip()
                # 다음 줄이 '이름' 후보 패턴과 일치하거나 씬 넘버로 시작하면 현재 줄은 이름이 아님
                is_next_line_name_candidate = bool(re.match(r'^([^\t\n]{1,30})$', next_line)) and not (next_line.endswith('는데-') or next_line.endswith('다.') or next_line.endswith('보인다.') or next_line.endswith('있다.'))
                is_scene_heading = next_line.startswith('#') or re.match(r'^\d+\.\s+', lines[next_line_index]) # 원본 줄 확인

                # 다음 줄이 새로운 이름 후보나 씬넘버가 아니라면 현재 줄을 이름으로 확정하고 대사 추출 시작
                if not is_next_line_name_candidate and not is_scene_heading:
                    character_name = potential_character_name # 이름 확정
                    current_dialogue_lines = []
                    dialogue_line_index = next_line_index

                    while dialogue_line_index < len(lines):
                        dialogue_part_raw = lines[dialogue_line_index] # 원본 줄 유지 (들여쓰기 등 확인 위함)
                        dialogue_part_stripped = dialogue_part_raw.strip()

                        # 다음 줄이 새로운 이름 후보거나 씬 넘버거나 빈 줄이면 대사 끝
                        is_dialogue_name_follow = bool(re.match(r'^([^\t\n]{1,30})$', dialogue_part_stripped)) and not (dialogue_part_stripped.endswith('는데-') or dialogue_part_stripped.endswith('다.') or dialogue_part_stripped.endswith('보인다.') or dialogue_part_stripped.endswith('있다.'))
                        is_scene_heading_follow = dialogue_part_stripped.startswith('#') or re.match(r'^\d+\.\s+', dialogue_part_raw)

                        if not dialogue_part_stripped or (is_dialogue_name_follow and not dialogue_part_raw.startswith('\t') and not dialogue_part_raw.startswith(' ')) or is_scene_heading_follow:
                             # 들여쓰기 없는 이름 후보는 새 이름으로 간주
                            break # 대사 끝

                        # 대사에 포함될 내용 (괄호 안 지문 제거)
                        cleaned_part = re.sub(r'\([^)]*\)', '', dialogue_part_stripped).strip()
                        if cleaned_part:
                            current_dialogue_lines.append(cleaned_part)
                        dialogue_line_index += 1

                    if current_dialogue_lines:
                        full_dialogue = " ".join(current_dialogue_lines).strip()
                        # 여러 공백을 하나로 변경
                        full_dialogue = ' '.join(full_dialogue.split())

                        if full_dialogue:
                            dialogues.append({
                                'Character': character_name,
                                'Dialogue': full_dialogue
                            })
                        i = dialogue_line_index # 다음 처리할 라인 인덱스 업데이트
                        continue # 다음 바깥 루프로

        # '이름 : 대사' 형식도 혹시 모르니 체크 (혼용 가능성)
        dialogue_colon_match = re.match(r'^\s*([^:\n]+?)\s*:\s*(.*)', line)
        if dialogue_colon_match:
             character_name = dialogue_colon_match.group(1).strip()
             dialogue_text = dialogue_colon_match.group(2).strip()
             cleaned_dialogue = re.sub(r'\([^)]*\)', '', dialogue_text).strip()
             cleaned_dialogue = ' '.join(cleaned_dialogue.split())
             if cleaned_dialogue:
                 # 이름만 있는 줄 다음 대사가 오는 형식을 우선하므로, 중복 방지 체크 필요 시 추가
                 # 예: if not dialogues or dialogues[-1]['Character'] != character_name:
                 dialogues.append({
                     'Character': character_name,
                     'Dialogue': cleaned_dialogue
                 })
             i += 1
             continue # 다음 루프로

        # 위 패턴에 모두 맞지 않으면 다음 줄로 이동
        i += 1

    return dialogues

In [ ]:
with open('./_data/1020/[대본]베테랑.txt', 'r', encoding='utf-8') as f:
    script_content = ''.join(f.readlines())

extracted_dialogues = extract_dialogue_enter_format(script_content)    



# 4. 결과 확인 (예: CSV 형태로 출력)
if extracted_dialogues:
    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_ALL)
    writer.writerow(['Character', 'Dialogue']) # 헤더 작성
    for entry in extracted_dialogues:
        writer.writerow([entry['Character'], entry['Dialogue']])

    csv_output = output.getvalue()
    output.close()
    print("--- CSV 출력 ---")
    print(csv_output)

    # 파일로 저장하고 싶을 경우:
    with open("./_data/1020/[대본]베테랑.csv", "w", encoding="utf-8", newline="") as f:
         f.write(csv_output)
    print("\nCSV 파일 '[대본]베테랑.csv' 생성 완료.")

else:
    print("추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.")        

추출된 대사가 없습니다. 스크립트 형식을 확인해주세요.
